<a href="https://colab.research.google.com/github/sselgrad23/Pandas-Tutorial-For-Beginners/blob/main/pandas_tutorial_lesson_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Handling Null Values

In [1]:
import pandas as pd
import numpy as np

In [2]:
def set_nan(idx_start, idx_end, column):
  """
  Sets the specific values in a given column to NaN for rows identified
  by their index labels, `idx_start` and `idx_end`.
  """
  coffee.loc[[idx_start,idx_end], column] = np.nan

In [3]:
coffee = pd.read_csv('https://raw.githubusercontent.com/KeithGalli/complete-pandas-tutorial/refs/heads/master/warmup-data/coffee.csv')
coffee.head()

,Day,Coffee Type,Units Sold
0,Monday,Espresso,25
1,Monday,Latte,15
2,Tuesday,Espresso,30
3,Tuesday,Latte,20
4,Wednesday,Espresso,35


In [4]:
coffee['price'] = np.where(coffee['Coffee Type'] == 'Espresso', 3.99, 5.99)
coffee['revenue'] = coffee['price'] * coffee['Units Sold']
coffee_backup = coffee.copy()
coffee.head()

,Day,Coffee Type,Units Sold,price,revenue
0,Monday,Espresso,25,3.99,99.75
1,Monday,Latte,15,5.99,89.85
2,Tuesday,Espresso,30,3.99,119.70
3,Tuesday,Latte,20,5.99,119.80
4,Wednesday,Espresso,35,3.99,139.65


Let's set two of the values to null. I use a function I have defined at the start of this noetbook:

In [5]:
set_nan(0, 1, 'Units Sold')
coffee.head()

,Day,Coffee Type,Units Sold,price,revenue
0,Monday,Espresso,NaN,3.99,99.75
1,Monday,Latte,NaN,5.99,89.85
2,Tuesday,Espresso,30.0,3.99,119.70
3,Tuesday,Latte,20.0,5.99,119.80
4,Wednesday,Espresso,35.0,3.99,139.65


.info() can be used to keep track of NaNs/null-values. For example, we can tell that 12 of the 14 Units Sold values are non-null:

In [6]:
coffee.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Day          14 non-null     object 
 1   Coffee Type  14 non-null     object 
 2   Units Sold   12 non-null     float64
 3   price        14 non-null     float64
 4   revenue      14 non-null     float64
dtypes: float64(3), object(2)
memory usage: 692.0+ bytes


But we can see more explicitly the number of NaN values by using .isna().sum():

In [7]:
coffee.isna().sum()

,0
Day,0
Coffee Type,0
Units Sold,2
price,0
revenue,0


pd.NA will also be counted by coffee.isna().sum():


In [8]:
coffee['Units Sold'] = coffee['Units Sold'].astype('Int64')  # capital I
coffee.loc[[0, 1], 'Units Sold'] = pd.NA
coffee.head()

,Day,Coffee Type,Units Sold,price,revenue
0,Monday,Espresso,<NA>,3.99,99.75
1,Monday,Latte,<NA>,5.99,89.85
2,Tuesday,Espresso,30,3.99,119.70
3,Tuesday,Latte,20,5.99,119.80
4,Wednesday,Espresso,35,3.99,139.65


In [9]:
coffee.isna().sum()

,0
Day,0
Coffee Type,0
Units Sold,2
price,0
revenue,0


But what if we don't have either of these, but have empty strings instead?

In [10]:
coffee['Units Sold'] = coffee['Units Sold'].astype(object)
coffee.loc[[0, 1], 'Units Sold'] = ''
display(coffee.head())
display(coffee.isna().sum())

,Day,Coffee Type,Units Sold,price,revenue
0,Monday,Espresso,,3.99,99.75
1,Monday,Latte,,5.99,89.85
2,Tuesday,Espresso,30,3.99,119.70
3,Tuesday,Latte,20,5.99,119.80
4,Wednesday,Espresso,35,3.99,139.65


,0
Day,0
Coffee Type,0
Units Sold,0
price,0
revenue,0


As you can see, `isna().sum()` reports 0 `NaN`s for 'Units Sold' because the empty strings are not treated as missing values.

One can use `.replace()` to convert empty strings to `np.nan`. However, this will result in the FutureWarning:

In [11]:
coffee['Units Sold'] = coffee['Units Sold'].replace('', np.nan).astype(object)
coffee['Units Sold'] = pd.to_numeric(coffee['Units Sold'], errors='coerce')

/tmp/ipykernel_22678/2677115822.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  coffee['Units Sold'] = coffee['Units Sold'].replace('', np.nan).astype(object)


To avoid the `FutureWarning` when dealing with empty strings, it's better to use `mask()` with `pd.to_numeric()`. `mask()` can replace values where a condition is True, effectively handling empty strings as missing values before converting to numeric types.

In [12]:
coffee['Units Sold'] = coffee['Units Sold'].astype(object)
coffee.loc[[0, 1], 'Units Sold'] = ''

coffee['Units Sold'] = pd.to_numeric(
    coffee['Units Sold'].mask(coffee['Units Sold'] == ''),
    errors='coerce'
)
display(coffee.head())

,Day,Coffee Type,Units Sold,price,revenue
0,Monday,Espresso,NaN,3.99,99.75
1,Monday,Latte,NaN,5.99,89.85
2,Tuesday,Espresso,30.0,3.99,119.70
3,Tuesday,Latte,20.0,5.99,119.80
4,Wednesday,Espresso,35.0,3.99,139.65


Now that the empty strings are replaced with `np.nan`, `isna().sum()` will correctly count them:

In [13]:
coffee.isna().sum()

,0
Day,0
Coffee Type,0
Units Sold,2
price,0
revenue,0


There are different ways of imputing missing values. Popular ways include imputing with the mean or median:

In [14]:
coffee.fillna(coffee['Units Sold'].mean()) # mean

,Day,Coffee Type,Units Sold,price,revenue
0,Monday,Espresso,35.0,3.99,99.75
1,Monday,Latte,35.0,5.99,89.85
2,Tuesday,Espresso,30.0,3.99,119.70
3,Tuesday,Latte,20.0,5.99,119.80
4,Wednesday,Espresso,35.0,3.99,139.65
5,Wednesday,Latte,25.0,5.99,149.75
6,Thursday,Espresso,40.0,3.99,159.60
7,Thursday,Latte,30.0,5.99,179.70
8,Friday,Espresso,45.0,3.99,179.55
9,Friday,Latte,35.0,5.99,209.65


In [15]:
set_nan(0, 1, 'Units Sold')
coffee.fillna(coffee['Units Sold'].median()) # median

,Day,Coffee Type,Units Sold,price,revenue
0,Monday,Espresso,35.0,3.99,99.75
1,Monday,Latte,35.0,5.99,89.85
2,Tuesday,Espresso,30.0,3.99,119.70
3,Tuesday,Latte,20.0,5.99,119.80
4,Wednesday,Espresso,35.0,3.99,139.65
5,Wednesday,Latte,25.0,5.99,149.75
6,Thursday,Espresso,40.0,3.99,159.60
7,Thursday,Latte,30.0,5.99,179.70
8,Friday,Espresso,45.0,3.99,179.55
9,Friday,Latte,35.0,5.99,209.65


In this particular example dataframe, the mean and median are the same. This can be confirmed:

In [16]:
# Calculate the mean and median of 'Units Sold' from the original DataFrame (before fillna)
set_nan(0, 1, 'Units Sold')
mean_units_sold = coffee['Units Sold'].mean()
median_units_sold = coffee['Units Sold'].median()

print(f"Mean of 'Units Sold': {mean_units_sold}")
print(f"Median of 'Units Sold': {median_units_sold}")

Mean of 'Units Sold': 35.0
Median of 'Units Sold': 35.0


Better yet, use describe(). 50% refers to the 50th percentile and is the median:

In [17]:
coffee.describe()

,Units Sold,price,revenue
count,12.00000,14.000000,14.000000
mean,35.00000,4.990000,158.957143
std,7.97724,1.037749,40.406967
min,20.00000,3.990000,89.850000
25%,30.00000,3.990000,124.762500
50%,35.00000,4.990000,169.575000
75%,41.25000,5.990000,179.662500
max,45.00000,5.990000,209.650000


Another way is to use .interpolate(). This takes the surrounding values into consideration and patterns in the data when imputing the missing values. However, it doesn't work when my missing values are at the 0th and 1st indices.

interpolate() uses linear interpolation by default. This requires at least two valid surrounding data points to fill in a missing value. If NaN values are at the very beginning of the Series, and there are no preceding non-missing values, interpolate() cannot fill them because there's no data before them to establish a trend. They will remain NaN.

In [18]:
set_nan(0, 1, 'Units Sold')
coffee['Units Sold'] = coffee['Units Sold'].interpolate()
coffee.head()

,Day,Coffee Type,Units Sold,price,revenue
0,Monday,Espresso,NaN,3.99,99.75
1,Monday,Latte,NaN,5.99,89.85
2,Tuesday,Espresso,30.0,3.99,119.70
3,Tuesday,Latte,20.0,5.99,119.80
4,Wednesday,Espresso,35.0,3.99,139.65


It does work when the NaNs are later in the column, though:

In [19]:
coffee = coffee_backup.copy()
set_nan(3, 4, 'Units Sold')
coffee['Units Sold'] = coffee['Units Sold'].interpolate()
coffee.head()

,Day,Coffee Type,Units Sold,price,revenue
0,Monday,Espresso,25.000000,3.99,99.75
1,Monday,Latte,15.000000,5.99,89.85
2,Tuesday,Espresso,30.000000,3.99,119.70
3,Tuesday,Latte,28.333333,5.99,119.80
4,Wednesday,Espresso,26.666667,3.99,139.65


Another method of dealing with rows with NaNs is simply to drop them using dropna().

While straightforward, this method should be used cautiously. If you are dealing with sparse dataframes or if there are many missing values, it is not recommended to drop all rows with NaNs as this can lead to a significant loss of valuable information and potentially introduce bias into your analysis.

In such cases, imputing with e.g. the mean, median, or using interpolation is often preferred to retain as much data as possible.

In [20]:
coffee = coffee_backup.copy()
set_nan(3, 4, 'Units Sold')
coffee = coffee.dropna()
display(coffee)

,Day,Coffee Type,Units Sold,price,revenue
0,Monday,Espresso,25.0,3.99,99.75
1,Monday,Latte,15.0,5.99,89.85
2,Tuesday,Espresso,30.0,3.99,119.70
5,Wednesday,Latte,25.0,5.99,149.75
6,Thursday,Espresso,40.0,3.99,159.60
7,Thursday,Latte,30.0,5.99,179.70
8,Friday,Espresso,45.0,3.99,179.55
9,Friday,Latte,35.0,5.99,209.65
10,Saturday,Espresso,45.0,3.99,179.55
11,Saturday,Latte,35.0,5.99,209.65


You can also choose to drop rows if specific columns have NaNs in them:

In [21]:
coffee = coffee_backup.copy()
set_nan(3, 4, 'Units Sold')
coffee.dropna(subset=['Units Sold'])

,Day,Coffee Type,Units Sold,price,revenue
0,Monday,Espresso,25.0,3.99,99.75
1,Monday,Latte,15.0,5.99,89.85
2,Tuesday,Espresso,30.0,3.99,119.70
5,Wednesday,Latte,25.0,5.99,149.75
6,Thursday,Espresso,40.0,3.99,159.60
7,Thursday,Latte,30.0,5.99,179.70
8,Friday,Espresso,45.0,3.99,179.55
9,Friday,Latte,35.0,5.99,209.65
10,Saturday,Espresso,45.0,3.99,179.55
11,Saturday,Latte,35.0,5.99,209.65


We can also find the rows with NaNs by doing the following:

In [22]:
coffee = coffee_backup.copy()
set_nan(3, 4, 'Units Sold')
coffee[coffee['Units Sold'].isna()]

,Day,Coffee Type,Units Sold,price,revenue
3,Tuesday,Latte,NaN,5.99,119.80
4,Wednesday,Espresso,NaN,3.99,139.65
